In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import time
import numpy as np

import jax
import jax.numpy as jnp
from jax import random, vmap, jacrev, hessian
from jax import tree_util
from functools import partial

# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# Geometry / time
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

# ============================================================
# Constants
# ============================================================
rho_f = DTYPE(11096.0)
rho_s = DTYPE(16020.0)

V_INLET_BC = DTYPE(0.4)
T_INLET = DTYPE(560.0)
P_OUTLET = DTYPE(0.0)
POWER_COEF = DTYPE(5e7)

FLUID_U_IC = DTYPE(0)
FLUID_V_IC = DTYPE(1e-12)
FLUID_T_IC = DTYPE(560.0)

v_neu   = DTYPE(2.416)
D_fuel  = DTYPE(0.008249)
D_fluid = DTYPE(0.01)

PHI_TOPBOT_VAL = DTYPE(0.5)
PHI_RIGHT_VAL  = DTYPE(0.0)

PHI_TXT_PATH = "./phi.txt"

# ============================================================
# Output map scales
# ============================================================
PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(300.0)
TF_OUT_SCALE  = DTYPE(150.0)
U_OUT_SCALE   = DTYPE(1.0)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(15.0)  # pressure scale fixed

PHI_VAL_SCALE = PHI_OUT_SCALE
TS_VAL_SCALE  = TS_OUT_SCALE
TF_VAL_SCALE  = TF_OUT_SCALE
U_VAL_SCALE   = U_OUT_SCALE
V_VAL_SCALE   = V_OUT_SCALE
P_VAL_SCALE   = P_OUT_SCALE
T_IF_VAL_SCALE = jnp.maximum(TS_VAL_SCALE, TF_VAL_SCALE)

# ============================================================
# Geometry scales
# ============================================================
Lx       = (x_max - x_min) + EPS
Ly_len   = (y_max - y_min) + EPS
Lx_fluid = (x_max - Ls) + EPS
Lx_solid = (Ls - x_min) + EPS
Lmin     = jnp.minimum(Lx, Ly_len)

GRAD_TSX_SCALE = TS_VAL_SCALE / (Lx_solid + EPS)
GRAD_TSY_SCALE = TS_VAL_SCALE / (Ly_len + EPS)
GRAD_TFX_SCALE = TF_VAL_SCALE / (Lx_fluid + EPS)
GRAD_TFY_SCALE = TF_VAL_SCALE / (Ly_len + EPS)
GRAD_UX_SCALE  = U_VAL_SCALE / (Lx_fluid + EPS)
GRAD_UY_SCALE  = U_VAL_SCALE / (Ly_len + EPS)
GRAD_VX_SCALE  = V_VAL_SCALE / (Lx_fluid + EPS)
GRAD_VY_SCALE  = V_VAL_SCALE / (Ly_len + EPS)
GRAD_PX_SCALE  = P_VAL_SCALE / (Lx_fluid + EPS)
GRAD_PY_SCALE  = P_VAL_SCALE / (Ly_len + EPS)

# ============================================================
# phi.txt
# ============================================================
def load_phi_piecewise_multilinear(path: str):
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    iy = lines.index("AXIS Y")
    it = lines.index("AXIS T")
    idata = lines.index("DATA")

    y_tokens = []
    k = iy + 1
    while k < len(lines) and lines[k] != "AXIS T":
        for tok in lines[k].split():
            try:
                float(tok)
                y_tokens.append(tok)
            except Exception:
                pass
        k += 1
    y = np.array([float(s) for s in y_tokens], dtype=np.float64)

    t_tokens = []
    k = it + 1
    while k < len(lines) and lines[k] != "DATA":
        for tok in lines[k].split():
            try:
                float(tok)
                t_tokens.append(tok)
            except Exception:
                pass
        k += 1
    t = np.array([float(s) for s in t_tokens], dtype=np.float64)

    data_tokens = []
    for ln in lines[idata + 1:]:
        for tok in ln.split():
            data_tokens.append(float(tok))
    data = np.array(data_tokens, dtype=np.float64)

    Z_t_y = data.reshape(len(t), len(y))
    Z_y_t = Z_t_y.T
    return y, t, Z_y_t

y_np, t_np, Z_np = load_phi_piecewise_multilinear(PHI_TXT_PATH)
PHI_Y = jnp.array(y_np, dtype=DTYPE)
PHI_T = jnp.array(t_np, dtype=DTYPE)
PHI_Z = jnp.array(Z_np, dtype=DTYPE)

@jax.jit
def phi_bc_yt(y, t, y_grid, t_grid, Z):
    Ny = y_grid.shape[0]
    Nt = t_grid.shape[0]

    iy = jnp.clip(jnp.searchsorted(y_grid, y, side="right") - 1, 0, Ny - 2)
    it = jnp.clip(jnp.searchsorted(t_grid, t, side="right") - 1, 0, Nt - 2)

    y0 = y_grid[iy]
    y1 = y_grid[iy + 1]
    t0 = t_grid[it]
    t1 = t_grid[it + 1]

    wy = (y - y0) / (y1 - y0 + EPS)
    wt = (t - t0) / (t1 - t0 + EPS)

    z00 = Z[iy, it]
    z10 = Z[iy + 1, it]
    z01 = Z[iy, it + 1]
    z11 = Z[iy + 1, it + 1]

    z0 = z00 + wy * (z10 - z00)
    z1 = z01 + wy * (z11 - z01)
    return z0 + wt * (z1 - z0)

# ============================================================
# Material laws
# ============================================================
def safe_T(T):
    return jnp.clip(T, DTYPE(300.0), DTYPE(1200.0))

def mu_f_fun(T):
    T = safe_T(T)
    return DTYPE(4.94e-4) * jnp.exp(DTYPE(754.1) / (T + EPS))

def k_f_fun(T):
    T = safe_T(T)
    return DTYPE(3.61) + DTYPE(1.517e-2) * T - DTYPE(1.741e-6) * T * T

def cp_f_fun(T):
    T = safe_T(T)
    return DTYPE(159.0) - DTYPE(2.72e-2) * T + DTYPE(7.12e-6) * T * T

def cp_s_fun(T):
    T = safe_T(T)
    return (DTYPE(1.359) + DTYPE(0.05812) * T + DTYPE(1.086e6) / (T * T + EPS)) * DTYPE(5.0)

def k_s_fun(T):
    T = safe_T(T)
    c0 = DTYPE(17.5) * (DTYPE(1.0) - DTYPE(0.223)) / (DTYPE(1.0) + DTYPE(0.161))
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c0 + c1 * T + c2 * T * T

def sigma_af_fuel(T):
    T = safe_T(T)
    return (
        DTYPE(2.416) * DTYPE(583.5) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
        - (DTYPE(13.47) * (T - DTYPE(560.0)) / (DTYPE(900.0) - DTYPE(560.0)) + DTYPE(7.53))
          * DTYPE(2.1479) * DTYPE(1.602)
        + DTYPE(0.185) * DTYPE(6.6072) * DTYPE(0.1)
        - DTYPE(680.9) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
    )

def sigma_af_fluid(T):
    T = safe_T(T)
    return -(DTYPE(20.0) + DTYPE(20.0) * (T - DTYPE(560.0)) / (DTYPE(800.0) - DTYPE(560.0)))

def dk_s_dT(T):
    T = safe_T(T)
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c1 + DTYPE(2.0) * c2 * T

def dk_f_dT(T):
    T = safe_T(T)
    return DTYPE(1.517e-2) - DTYPE(2.0) * DTYPE(1.741e-6) * T

def dmu_f_dT(T):
    T = safe_T(T)
    mu = mu_f_fun(T)
    return mu * (-DTYPE(754.1)) / (T * T + EPS)

def div_k_grad_scalar(k, dk_dT_fun, T, Tx, Ty, Txx, Tyy):
    return k * (Txx + Tyy) + dk_dT_fun(T) * (Tx * Tx + Ty * Ty)

def div_mu_grad_component(mu, dmu_dT_fun, Tf, Tfx, Tfy, ux, uy, uxx, uyy):
    return mu * (uxx + uyy) + dmu_dT_fun(Tf) * (Tfx * ux + Tfy * uy)

# ============================================================
# Residual scales
# ============================================================
D_MAX   = jnp.maximum(D_fuel, D_fluid)
SIG_REF = jnp.maximum(jnp.abs(sigma_af_fuel(T_INLET)), jnp.abs(sigma_af_fluid(T_INLET)))
K_S_REF = k_s_fun(T_INLET)
K_F_REF = k_f_fun(T_INLET)
CP_S_REF = cp_s_fun(T_INLET)
CP_F_REF = cp_f_fun(T_INLET)
MU_REF   = mu_f_fun(T_INLET)

FLUX_SCALE = jnp.maximum(
    K_S_REF * TS_VAL_SCALE / (Lx_solid + EPS),
    K_F_REF * TF_VAL_SCALE / (Lx_fluid + EPS),
) + EPS

PHI_TIME_REF = PHI_VAL_SCALE / (v_neu * (T_end + EPS))
PHI_DIFF_REF = D_MAX * PHI_VAL_SCALE / (Lmin**2 + EPS)
PHI_REAC_REF = SIG_REF * PHI_VAL_SCALE
PHI_RES_SCALE = PHI_TIME_REF + PHI_DIFF_REF + PHI_REAC_REF + EPS

TS_TIME_REF = rho_s * CP_S_REF * TS_VAL_SCALE / (T_end + EPS)
TS_DIFF_REF = K_S_REF * TS_VAL_SCALE / (Lx_solid**2 + EPS)
TS_SRC_REF  = POWER_COEF * PHI_VAL_SCALE
TS_RES_SCALE = TS_TIME_REF + TS_DIFF_REF + TS_SRC_REF + EPS

CONT_SCALE = jnp.maximum(U_VAL_SCALE / (Lx_fluid + EPS), V_VAL_SCALE / (Ly_len + EPS)) + EPS

MOMX_TIME_REF = rho_f * U_VAL_SCALE / (T_end + EPS)
MOMX_ADV_REF  = rho_f * (U_VAL_SCALE * U_VAL_SCALE / (Lx_fluid + EPS) + V_VAL_SCALE * U_VAL_SCALE / (Ly_len + EPS))
MOMX_P_REF    = P_VAL_SCALE / (Lx_fluid + EPS)
MOMX_VIS_REF  = MU_REF * (U_VAL_SCALE / (Lx_fluid**2 + EPS) + U_VAL_SCALE / (Ly_len**2 + EPS))
MOMX_RES_SCALE = MOMX_TIME_REF + MOMX_ADV_REF + MOMX_P_REF + MOMX_VIS_REF + EPS

MOMY_TIME_REF = rho_f * V_VAL_SCALE / (T_end + EPS)
MOMY_ADV_REF  = rho_f * (U_VAL_SCALE * V_VAL_SCALE / (Lx_fluid + EPS) + V_VAL_SCALE * V_VAL_SCALE / (Ly_len + EPS))
MOMY_P_REF    = P_VAL_SCALE / (Ly_len + EPS)
MOMY_VIS_REF  = MU_REF * (V_VAL_SCALE / (Lx_fluid**2 + EPS) + V_VAL_SCALE / (Ly_len**2 + EPS))
MOMY_RES_SCALE = MOMY_TIME_REF + MOMY_ADV_REF + MOMY_P_REF + MOMY_VIS_REF + EPS

TF_TIME_REF = rho_f * CP_F_REF * TF_VAL_SCALE / (T_end + EPS)
TF_ADV_REF  = rho_f * CP_F_REF * (
    U_VAL_SCALE * TF_VAL_SCALE / (Lx_fluid + EPS) +
    V_VAL_SCALE * TF_VAL_SCALE / (Ly_len + EPS)
)
TF_DIFF_REF = K_F_REF * TF_VAL_SCALE * (1.0 / (Lx_fluid**2 + EPS) + 1.0 / (Ly_len**2 + EPS))
TF_RES_SCALE = TF_TIME_REF + TF_ADV_REF + TF_DIFF_REF + EPS

# ============================================================
# Targets
# ============================================================
@jax.jit
def phi_ic(y):
    return DTYPE(2.0) * jnp.cos(DTYPE(3.14) * (y - DTYPE(0.375)) / DTYPE(0.75))

def phi_left_target(y, t):
    t_clip = jnp.clip(t, PHI_T[0], PHI_T[-1])
    return phi_bc_yt(y, t_clip, PHI_Y, PHI_T, PHI_Z)

def phi_right_target(y, t):
    return DTYPE(0.0) * jnp.ones_like(y)

def phi_y_target(y_fixed, t):
    return PHI_TOPBOT_VAL * jnp.ones_like(t)

def inlet_v_profile(x):
    return V_INLET_BC * jnp.ones_like(x)

# ============================================================
# Input normalization
# ============================================================
def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]
    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

# ============================================================
# Network
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def forward6(params, xyt):
    return mlp_apply(params, xyt[None, :])[0, :]

# ============================================================
# Sampling
# ============================================================
def sample_box(key, n, x0, x1, y0, y1, t0, t1):
    k1, k2, k3 = random.split(key, 3)
    x = random.uniform(k1, (n,1), minval=DTYPE(x0), maxval=DTYPE(x1), dtype=DTYPE)
    y = random.uniform(k2, (n,1), minval=DTYPE(y0), maxval=DTYPE(y1), dtype=DTYPE)
    t = random.uniform(k3, (n,1), minval=DTYPE(t0), maxval=DTYPE(t1), dtype=DTYPE)
    return jnp.concatenate([x, y, t], axis=1)

def sample_boundary_x(key, n, x_fixed, y0, y1, t0, t1):
    k1, k2 = random.split(key, 2)
    y = random.uniform(k1, (n,1), minval=DTYPE(y0), maxval=DTYPE(y1), dtype=DTYPE)
    t = random.uniform(k2, (n,1), minval=DTYPE(t0), maxval=DTYPE(t1), dtype=DTYPE)
    x = DTYPE(x_fixed) * jnp.ones_like(y)
    return jnp.concatenate([x, y, t], axis=1)

def sample_boundary_y(key, n, y_fixed, x0, x1, t0, t1):
    k1, k2 = random.split(key, 2)
    x = random.uniform(k1, (n,1), minval=DTYPE(x0), maxval=DTYPE(x1), dtype=DTYPE)
    t = random.uniform(k2, (n,1), minval=DTYPE(t0), maxval=DTYPE(t1), dtype=DTYPE)
    y = DTYPE(y_fixed) * jnp.ones_like(x)
    return jnp.concatenate([x, y, t], axis=1)

def sample_interface(key, n):
    return sample_boundary_x(key, n, Ls, y_min, y_max, t_min, t_max)

def sample_ic(key, n):
    k1, k2 = random.split(key, 2)
    x = random.uniform(k1, (n,1), minval=x_min, maxval=x_max, dtype=DTYPE)
    y = random.uniform(k2, (n,1), minval=y_min, maxval=y_max, dtype=DTYPE)
    t = t_min * jnp.ones_like(x)
    return jnp.concatenate([x, y, t], axis=1)

# ============================================================
# Derivatives / residuals
# ============================================================
def eval_fields_and_derivs(params, X):
    out = vmap(lambda z: forward6(params, z))(X)
    J   = vmap(jacrev(lambda z: forward6(params, z)))(X)
    H   = vmap(hessian(lambda z: forward6(params, z)))(X)
    return out, J, H

def residuals_all(params, X):
    out, J, H = eval_fields_and_derivs(params, X)

    phi = out[:, 0]
    Ts  = out[:, 1]
    u   = out[:, 2]
    v   = out[:, 3]
    p   = out[:, 4]
    Tf  = out[:, 5]

    x = X[:, 0]
    m_s = (x <= Ls).astype(DTYPE)
    m_f = (x >  Ls).astype(DTYPE)

    phi_t  = J[:, 0, 2]
    phi_xx = H[:, 0, 0, 0]
    phi_yy = H[:, 0, 1, 1]
    lap_phi = phi_xx + phi_yy

    Ts_x  = J[:, 1, 0]
    Ts_y  = J[:, 1, 1]
    Ts_t  = J[:, 1, 2]
    Ts_xx = H[:, 1, 0, 0]
    Ts_yy = H[:, 1, 1, 1]

    u_x, u_y, u_t = J[:, 2, 0], J[:, 2, 1], J[:, 2, 2]
    v_x, v_y, v_t = J[:, 3, 0], J[:, 3, 1], J[:, 3, 2]
    p_x, p_y      = J[:, 4, 0], J[:, 4, 1]
    Tf_x, Tf_y, Tf_t = J[:, 5, 0], J[:, 5, 1], J[:, 5, 2]

    u_xx, u_yy   = H[:, 2, 0, 0], H[:, 2, 1, 1]
    v_xx, v_yy   = H[:, 3, 0, 0], H[:, 3, 1, 1]
    Tf_xx, Tf_yy = H[:, 5, 0, 0], H[:, 5, 1, 1]

    is_solid = (x <= Ls)
    T_for_sigma = jnp.where(is_solid, Ts, Tf)
    D_here      = jnp.where(is_solid, D_fuel, D_fluid)
    sig_here    = jnp.where(is_solid, sigma_af_fuel(T_for_sigma), sigma_af_fluid(T_for_sigma))

    r_phi_raw = (DTYPE(1.0) / v_neu) * phi_t - D_here * lap_phi - sig_here * phi
    r_phi = r_phi_raw / PHI_RES_SCALE

    div_k_grad_Ts = div_k_grad_scalar(k_s_fun(Ts), dk_s_dT, Ts, Ts_x, Ts_y, Ts_xx, Ts_yy)
    r_Ts_raw = rho_s * cp_s_fun(Ts) * Ts_t - div_k_grad_Ts - POWER_COEF * phi
    r_Ts = r_Ts_raw / TS_RES_SCALE

    r_cont_raw = u_x + v_y
    r_cont = r_cont_raw / CONT_SCALE

    mu_here = mu_f_fun(Tf)
    visc_u = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, u_x, u_y, u_xx, u_yy)
    visc_v = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, v_x, v_y, v_xx, v_yy)

    r_u_raw = rho_f * (u_t + u * u_x + v * u_y) + p_x - visc_u
    r_v_raw = rho_f * (v_t + u * v_x + v * v_y) + p_y - visc_v

    cp_here = cp_f_fun(Tf)
    div_k_grad_Tf = div_k_grad_scalar(k_f_fun(Tf), dk_f_dT, Tf, Tf_x, Tf_y, Tf_xx, Tf_yy)
    r_Tf_raw = rho_f * cp_here * (Tf_t + u * Tf_x + v * Tf_y) - div_k_grad_Tf

    r_u  = r_u_raw / MOMX_RES_SCALE
    r_v  = r_v_raw / MOMY_RES_SCALE
    r_Tf = r_Tf_raw / TF_RES_SCALE

    return r_phi, r_Ts, r_cont, r_u, r_v, r_Tf, m_s, m_f

# ============================================================
# Loss pieces
# ============================================================
def mse(x):
    return jnp.mean(x * x)

def pde_loss(params, X_phi, Xs, Xf, w_pde_phi, w_pde_ts, w_pde_cont, w_pde_ru, w_pde_rv, w_pde_rtf):
    rphi_all, _, _, _, _, _, _, _ = residuals_all(params, X_phi)
    _, rTs_s, _, _, _, _, _, _ = residuals_all(params, Xs)
    _, _, rcont_f, ru_f, rv_f, rTf_f, _, _ = residuals_all(params, Xf)

    lpde_phi = mse(rphi_all)
    lpde_s   = mse(rTs_s)
    lcont    = mse(rcont_f)
    lru      = mse(ru_f)
    lrv      = mse(rv_f)
    lrtf     = mse(rTf_f)

    total = (
        DTYPE(w_pde_phi) * lpde_phi
        + DTYPE(w_pde_ts) * lpde_s
        + DTYPE(w_pde_cont) * lcont
        + DTYPE(w_pde_ru) * lru
        + DTYPE(w_pde_rv) * lrv
        + DTYPE(w_pde_rtf) * lrtf
    )
    return total, (lpde_phi, lpde_s, lcont, lru, lrv, lrtf)

def bc_phi_loss(params, X_left, X_right, X_bot, X_top):
    outL = mlp_apply(params, X_left)
    outR = mlp_apply(params, X_right)
    outB = mlp_apply(params, X_bot)
    outT = mlp_apply(params, X_top)

    phiL = outL[:,0]
    phiR = outR[:,0]
    phiB = outB[:,0]
    phiT = outT[:,0]

    yL, tL = X_left[:,1], X_left[:,2]
    yR, tR = X_right[:,1], X_right[:,2]
    tB = X_bot[:,2]
    tT = X_top[:,2]

    return (
        mse((phiL - phi_left_target(yL, tL)) / PHI_VAL_SCALE)
        + mse((phiR - phi_right_target(yR, tR)) / PHI_VAL_SCALE)
        + mse((phiB - phi_y_target(y_min, tB)) / PHI_VAL_SCALE)
        + mse((phiT - phi_y_target(y_max, tT)) / PHI_VAL_SCALE)
    )

def bc_solid_loss(params, X_x0, X_y0, X_y1):
    def Ts_fun(z): return forward6(params, z)[1]
    Tsx_x0 = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_x0)
    Tsy_y0 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_y0)
    Tsy_y1 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_y1)
    return mse(Tsx_x0 / GRAD_TSX_SCALE) + mse(Tsy_y0 / GRAD_TSY_SCALE) + mse(Tsy_y1 / GRAD_TSY_SCALE)

def bc_fluid_loss(params, X_inlet, X_outlet, X_right):
    out_in = mlp_apply(params, X_inlet)
    out_out = mlp_apply(params, X_outlet)
    out_r = mlp_apply(params, X_right)

    u_in, v_in, Tf_in = out_in[:,2], out_in[:,3], out_in[:,5]
    p_out = out_out[:,4]
    u_r = out_r[:,2]

    def u_fun(z): return forward6(params, z)[2]
    def v_fun(z): return forward6(params, z)[3]
    def p_fun(z): return forward6(params, z)[4]
    def Tf_fun(z): return forward6(params, z)[5]

    uy_out  = vmap(lambda z: jacrev(u_fun)(z)[1])(X_outlet)
    vy_out  = vmap(lambda z: jacrev(v_fun)(z)[1])(X_outlet)
    Tfy_out = vmap(lambda z: jacrev(Tf_fun)(z)[1])(X_outlet)

    vx_r  = vmap(lambda z: jacrev(v_fun)(z)[0])(X_right)
    px_r  = vmap(lambda z: jacrev(p_fun)(z)[0])(X_right)
    Tfx_r = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_right)

    x_in = X_inlet[:,0]
    v_tar = inlet_v_profile(x_in)

    return (
        mse((u_in - DTYPE(0.0)) / U_VAL_SCALE)
        + mse((v_in - v_tar) / V_VAL_SCALE)
        + mse((Tf_in - T_INLET) / TF_VAL_SCALE)
        + mse((p_out - P_OUTLET) / P_VAL_SCALE)
        + mse(uy_out / GRAD_UY_SCALE)
        + mse(vy_out / GRAD_VY_SCALE)
        + mse(Tfy_out / GRAD_TFY_SCALE)
        + mse(u_r / U_VAL_SCALE)
        + mse(vx_r / GRAD_VX_SCALE)
 
    )

def interface_loss(params, X_if):
    out_if = mlp_apply(params, X_if)
    Ts_if, Tf_if = out_if[:,1], out_if[:,5]

    def Ts_fun(z): return forward6(params, z)[1]
    def Tf_fun(z): return forward6(params, z)[5]
    def u_fun(z):  return forward6(params, z)[2]
    def v_fun(z):  return forward6(params, z)[3]

    Tsx_if = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_if)
    Tfx_if = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_if)

    q_s = -k_s_fun(Ts_if) * Tsx_if
    q_f = -k_f_fun(Tf_if) * Tfx_if

    u_if = vmap(lambda z: u_fun(z))(X_if)
    v_if = vmap(lambda z: v_fun(z))(X_if)

    return (
        mse((Ts_if - Tf_if) / T_IF_VAL_SCALE)
        + mse((q_s - q_f) / FLUX_SCALE)
        + mse(u_if / U_VAL_SCALE)
        + mse(v_if / V_VAL_SCALE)
    )

def ic_loss(params, X_ic):
    out0 = mlp_apply(params, X_ic)
    phi0 = out0[:,0]
    Ts0  = out0[:,1]
    u0   = out0[:,2]
    v0   = out0[:,3]
    Tf0  = out0[:,5]

    x0 = X_ic[:,0]
    y0 = X_ic[:,1]
    m_s0 = (x0 <= Ls).astype(DTYPE)
    m_f0 = (x0 >  Ls).astype(DTYPE)

    return (
        mse((phi0 - phi_ic(y0)) / PHI_VAL_SCALE)
        + mse(m_s0 * ((Ts0 - T_INLET) / TS_VAL_SCALE))
        + mse(m_f0 * ((u0  - FLUID_U_IC) / U_VAL_SCALE))
        + mse(m_f0 * ((v0  - FLUID_V_IC) / V_VAL_SCALE))
        + mse(m_f0 * ((Tf0 - FLUID_T_IC) / TF_VAL_SCALE))
    )

def pressure_anchor_loss(params, Xf):
    p_f = mlp_apply(params, Xf)[:, 4]
    return mse((p_f - jnp.mean(p_f)) / P_VAL_SCALE)

# ============================================================
# Total loss
# ============================================================
@partial(jax.jit, static_argnames=())
def loss_fn(
    params,
    X_pde_phi, X_pde_s, X_pde_f,
    X_phi_left, X_phi_right, X_phi_y0, X_phi_y1,
    X_Ts_x0, X_Ts_y0, X_Ts_y1,
    X_f_inlet, X_f_outlet, X_f_right,
    X_if, X_ic,
    w_pde_phi, w_pde_ts, w_pde_cont, w_pde_ru, w_pde_rv, w_pde_rtf,
    w_bc_phi, w_bc_solid, w_bc_fluid, w_if, w_ic, w_p_anchor
):
    lpde_total, (lpde_phi, lpde_s, lcont, lru, lrv, lrtf) = pde_loss(
        params, X_pde_phi, X_pde_s, X_pde_f,
        w_pde_phi, w_pde_ts, w_pde_cont, w_pde_ru, w_pde_rv, w_pde_rtf
    )

    lbc_phi = bc_phi_loss(params, X_phi_left, X_phi_right, X_phi_y0, X_phi_y1)
    lbc_sol = bc_solid_loss(params, X_Ts_x0, X_Ts_y0, X_Ts_y1)
    lbc_fld = bc_fluid_loss(params, X_f_inlet, X_f_outlet, X_f_right)
    lif = interface_loss(params, X_if)
    lic = ic_loss(params, X_ic)
    

    total = (
        lpde_total
        + DTYPE(w_bc_phi) * lbc_phi
        + DTYPE(w_bc_solid) * lbc_sol
        + DTYPE(w_bc_fluid) * lbc_fld
        + DTYPE(w_if) * lif
        + DTYPE(w_ic) * lic
        
    )

    return total, (lpde_phi, lpde_s, lcont, lru, lrv, lrtf, lbc_phi, lbc_sol, lbc_fld, lif, lic, lp_anchor)

loss_and_grad = jax.jit(jax.value_and_grad(loss_fn, argnums=0, has_aux=True))

# ============================================================
# Adam
# ============================================================
def adam_init(params):
    m = tree_util.tree_map(lambda p: jnp.zeros_like(p), params)
    v = tree_util.tree_map(lambda p: jnp.zeros_like(p), params)
    return m, v

def adam_step(params, grads, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m = tree_util.tree_map(lambda m_i, g_i: beta1 * m_i + (1.0 - beta1) * g_i, m, grads)
    v = tree_util.tree_map(lambda v_i, g_i: beta2 * v_i + (1.0 - beta2) * (g_i * g_i), v, grads)

    mhat = tree_util.tree_map(lambda m_i: m_i / (1.0 - beta1 ** t), m)
    vhat = tree_util.tree_map(lambda v_i: v_i / (1.0 - beta2 ** t), v)

    params = tree_util.tree_map(
        lambda p_i, mh_i, vh_i: p_i - lr * mh_i / (jnp.sqrt(vh_i) + eps),
        params, mhat, vhat
    )
    return params, m, v

def flatten_params(params):
    flat = []
    for lyr in params:
        flat.append(np.asarray(lyr["W"]).reshape(-1))
        flat.append(np.asarray(lyr["b"]).reshape(-1))
    return np.concatenate(flat)

def _rms(x):
    return float(jnp.sqrt(jnp.mean(x * x) + 1e-30))

# ============================================================
# Train
# ============================================================
def train_adam(
    seed=0,
    hidden_dim=64,
    num_hidden=3,
    max_iters=5000,
    print_every=100,
    lr=1e-4,

    n_pde_phi=2000,
    n_pde_s=2000,
    n_pde_f=4000,
    n_bc_each=512,
    n_if=1024,
    n_ic=1024,

    w_pde_phi=10.0,
    w_pde_ts=20.0,
    w_pde_cont=10.0,
    w_pde_ru=20.0,
    w_pde_rv=40.0,
    w_pde_rtf=5.0,

    w_bc_phi=10.0,
    w_bc_solid=2.0,
    w_bc_fluid=8.0,
    w_if=8.0,
    w_ic=8.0,
    w_p_anchor=0,
):
    key = random.PRNGKey(seed)

    layer_sizes = [3] + [hidden_dim] * num_hidden + [6]
    key, k0 = random.split(key)
    params = init_mlp_params(k0, layer_sizes)

    m, v = adam_init(params)

    print("Adam training")
    print("layer_sizes =", layer_sizes)
    print("P_OUT_SCALE =", float(P_OUT_SCALE))

    t0_clock = time.time()

    for it in range(1, max_iters + 1):
        key, *sub = random.split(key, 16)
        (
            kphi, ks, kf,
            k1, k2, k3, k4,
            k5, k6, k7,
            k8, k9, k10,
            kif, kic
        ) = sub

        X_pde_phi = sample_box(kphi, n_pde_phi, x_min, x_max, y_min, y_max, t_min, t_max)
        X_pde_s   = sample_box(ks,   n_pde_s,   x_min, Ls,   y_min, y_max, t_min, t_max)
        X_pde_f   = sample_box(kf,   n_pde_f,   Ls,   x_max, y_min, y_max, t_min, t_max)

        X_phi_left  = sample_boundary_x(k1, n_bc_each, x_min, y_min, y_max, t_min, t_max)
        X_phi_right = sample_boundary_x(k2, n_bc_each, x_max, y_min, y_max, t_min, t_max)
        X_phi_y0    = sample_boundary_y(k3, n_bc_each, y_min, x_min, x_max, t_min, t_max)
        X_phi_y1    = sample_boundary_y(k4, n_bc_each, y_max, x_min, x_max, t_min, t_max)

        X_Ts_x0 = sample_boundary_x(k5, n_bc_each, x_min, y_min, y_max, t_min, t_max)
        X_Ts_y0 = sample_boundary_y(k6, n_bc_each, y_min, x_min, Ls,   t_min, t_max)
        X_Ts_y1 = sample_boundary_y(k7, n_bc_each, y_max, x_min, Ls,   t_min, t_max)

        X_f_inlet  = sample_boundary_y(k8,  n_bc_each, y_min, Ls,   x_max, t_min, t_max)
        X_f_outlet = sample_boundary_y(k9,  n_bc_each, y_max, Ls,   x_max, t_min, t_max)
        X_f_right  = sample_boundary_x(k10, n_bc_each, x_max, y_min, y_max, t_min, t_max)

        X_if = sample_interface(kif, n_if)
        X_ic = sample_ic(kic, n_ic)

        (loss_val, aux), grads = loss_and_grad(
            params,
            X_pde_phi, X_pde_s, X_pde_f,
            X_phi_left, X_phi_right, X_phi_y0, X_phi_y1,
            X_Ts_x0, X_Ts_y0, X_Ts_y1,
            X_f_inlet, X_f_outlet, X_f_right,
            X_if, X_ic,
            DTYPE(w_pde_phi), DTYPE(w_pde_ts), DTYPE(w_pde_cont), DTYPE(w_pde_ru), DTYPE(w_pde_rv), DTYPE(w_pde_rtf),
            DTYPE(w_bc_phi), DTYPE(w_bc_solid), DTYPE(w_bc_fluid), DTYPE(w_if), DTYPE(w_ic), DTYPE(w_p_anchor)
        )

        params, m, v = adam_step(params, grads, m, v, it, lr)

        if it % print_every == 0:
            lpde_phi, lpde_s, lcont, lru, lrv, lrtf, lbc_phi, lbc_sol, lbc_fld, lif, lic= aux

            rphi_all, _, _, _, _, _, _, _ = residuals_all(params, X_pde_phi)
            _, rTs_s, _, _, _, _, _, _ = residuals_all(params, X_pde_s)
            _, _, rcont_f, ru_f, rv_f, rTf_f, _, _ = residuals_all(params, X_pde_f)

            print(
                f"[Adam it={it}] loss={float(loss_val):.3e} "
                f"lpde_phi={float(lpde_phi):.3e} lpde_s={float(lpde_s):.3e} "
                f"cont={float(lcont):.3e} ru={float(lru):.3e} rv={float(lrv):.3e} rTf={float(lrtf):.3e} "
                f"lbc_phi={float(lbc_phi):.3e} lbc_sol={float(lbc_sol):.3e} "
                f"lbc_fld={float(lbc_fld):.3e} lif={float(lif):.3e} lic={float(lic):.3e} "
                f"lp_anchor={float(lp_anchor):.3e}"
            )
            print(
                "Scaled PDE RMS:",
                "phi", _rms(rphi_all),
                "Ts_s", _rms(rTs_s),
                "cont_f", _rms(rcont_f),
                "ru_f", _rms(ru_f),
                "rv_f", _rms(rv_f),
                "rTf_f", _rms(rTf_f),
            )

    print(f"[done] elapsed={time.time() - t0_clock:.2f}s")
    return params

# ============================================================
# Main
# ============================================================
def main():
    params = train_adam()
    theta = flatten_params(params)
    np.save("adam_pinn_params_restart.npy", theta)
    print("Saved flat theta -> adam_pinn_params_restart.npy")

if __name__ == "__main__":
    main()

Adam training
layer_sizes = [3, 64, 64, 64, 6]
P_OUT_SCALE = 15.0


NameError: name 'lp_anchor' is not defined

: 

In [3]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import time
import math
import numpy as np

import jax
import jax.numpy as jnp
from jax import random, vmap, jacrev, hessian
from jax import tree_util
from functools import partial

# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# Geometry / time
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

# ============================================================
# Constants
# ============================================================
rho_f = DTYPE(11096.0)
rho_s = DTYPE(16020.0)

V_INLET_BC = DTYPE(0.4)
T_INLET = DTYPE(560.0)
P_OUTLET = DTYPE(0.0)
POWER_COEF = DTYPE(5e7)

# match current SQP benchmark
FLUID_U_IC = DTYPE(1.0)
FLUID_V_IC = DTYPE(1e-12)
FLUID_T_IC = DTYPE(560.0)

v_neu   = DTYPE(2.416)
D_fuel  = DTYPE(0.008249)
D_fluid = DTYPE(0.01)

PHI_TOPBOT_VAL = DTYPE(0.5)
PHI_RIGHT_VAL  = DTYPE(0.0)

PHI_TXT_PATH = "./phi.txt"

# ============================================================
# Output scales
# match current SQP benchmark
# ============================================================
PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(350.0)
TF_OUT_SCALE  = DTYPE(250.0)
U_OUT_SCALE   = DTYPE(1.0)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(7.0)

PHI_VAL_SCALE = PHI_OUT_SCALE
TS_VAL_SCALE  = TS_OUT_SCALE
TF_VAL_SCALE  = TF_OUT_SCALE
U_VAL_SCALE   = U_OUT_SCALE
V_VAL_SCALE   = V_OUT_SCALE
P_VAL_SCALE   = P_OUT_SCALE
T_IF_VAL_SCALE = jnp.maximum(TS_VAL_SCALE, TF_VAL_SCALE)

# ============================================================
# Geometry scales
# ============================================================
Lx       = (x_max - x_min) + EPS
Ly_len   = (y_max - y_min) + EPS
Lx_fluid = (x_max - Ls) + EPS
Lx_solid = (Ls - x_min) + EPS
Lmin     = jnp.minimum(Lx, Ly_len)

GRAD_TSX_SCALE = TS_VAL_SCALE / (Lx_solid + EPS)

# ============================================================
# phi.txt
# ============================================================
def load_phi_piecewise_multilinear(path: str):
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    iy = lines.index("AXIS Y")
    it = lines.index("AXIS T")
    idata = lines.index("DATA")

    y_tokens = []
    k = iy + 1
    while k < len(lines) and lines[k] != "AXIS T":
        for tok in lines[k].split():
            try:
                float(tok)
                y_tokens.append(tok)
            except Exception:
                pass
        k += 1
    y = np.array([float(s) for s in y_tokens], dtype=np.float64)

    t_tokens = []
    k = it + 1
    while k < len(lines) and lines[k] != "DATA":
        for tok in lines[k].split():
            try:
                float(tok)
                t_tokens.append(tok)
            except Exception:
                pass
        k += 1
    t = np.array([float(s) for s in t_tokens], dtype=np.float64)

    data_tokens = []
    for ln in lines[idata + 1:]:
        for tok in ln.split():
            data_tokens.append(float(tok))
    data = np.array(data_tokens, dtype=np.float64)

    Z_t_y = data.reshape(len(t), len(y))
    Z_y_t = Z_t_y.T
    return y, t, Z_y_t

y_np, t_np, Z_np = load_phi_piecewise_multilinear(PHI_TXT_PATH)
PHI_Y = jnp.array(y_np, dtype=DTYPE)
PHI_T = jnp.array(t_np, dtype=DTYPE)
PHI_Z = jnp.array(Z_np, dtype=DTYPE)

@jax.jit
def interp_yt(y, t, y_grid, t_grid, Z):
    Ny = y_grid.shape[0]
    Nt = t_grid.shape[0]

    iy = jnp.clip(jnp.searchsorted(y_grid, y, side="right") - 1, 0, Ny - 2)
    it = jnp.clip(jnp.searchsorted(t_grid, t, side="right") - 1, 0, Nt - 2)

    y0 = y_grid[iy]
    y1 = y_grid[iy + 1]
    t0 = t_grid[it]
    t1 = t_grid[it + 1]

    wy = (y - y0) / (y1 - y0 + EPS)
    wt = (t - t0) / (t1 - t0 + EPS)

    z00 = Z[iy, it]
    z10 = Z[iy + 1, it]
    z01 = Z[iy, it + 1]
    z11 = Z[iy + 1, it + 1]

    z0 = z00 + wy * (z10 - z00)
    z1 = z01 + wy * (z11 - z01)
    return z0 + wt * (z1 - z0)

def phi_left_target(y, t):
    t_clip = jnp.clip(t, PHI_T[0], PHI_T[-1])
    return interp_yt(y, t_clip, PHI_Y, PHI_T, PHI_Z)

def phi_right_target(y, t):
    return DTYPE(0.0) * jnp.ones_like(y)

def phi_y_target(t):
    return PHI_TOPBOT_VAL * jnp.ones_like(t)

# ============================================================
# Material laws
# ============================================================
def safe_T(T):
    return jnp.clip(T, DTYPE(300.0), DTYPE(1200.0))

def mu_f_fun(T):
    T = safe_T(T)
    return DTYPE(4.94e-4) * jnp.exp(DTYPE(754.1) / (T + EPS))

def k_f_fun(T):
    T = safe_T(T)
    return DTYPE(3.61) + DTYPE(1.517e-2) * T - DTYPE(1.741e-6) * T * T

def cp_f_fun(T):
    T = safe_T(T)
    return DTYPE(159.0) - DTYPE(2.72e-2) * T + DTYPE(7.12e-6) * T * T

def cp_s_fun(T):
    T = safe_T(T)
    return (DTYPE(1.359) + DTYPE(0.05812) * T + DTYPE(1.086e6) / (T * T + EPS)) * DTYPE(5.0)

def k_s_fun(T):
    T = safe_T(T)
    c0 = DTYPE(17.5) * (DTYPE(1.0) - DTYPE(0.223)) / (DTYPE(1.0) + DTYPE(0.161))
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c0 + c1 * T + c2 * T * T

def sigma_af_fuel(T):
    T = safe_T(T)
    return (
        DTYPE(2.416) * DTYPE(583.5) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
        - (DTYPE(13.47) * (T - DTYPE(560.0)) / (DTYPE(900.0) - DTYPE(560.0)) + DTYPE(7.53))
          * DTYPE(2.1479) * DTYPE(1.602)
        + DTYPE(0.185) * DTYPE(6.6072) * DTYPE(0.1)
        - DTYPE(680.9) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
    )

def sigma_af_fluid(T):
    T = safe_T(T)
    return -(DTYPE(20.0) + DTYPE(20.0) * (T - DTYPE(560.0)) / (DTYPE(800.0) - DTYPE(560.0)))

def dk_s_dT(T):
    T = safe_T(T)
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c1 + DTYPE(2.0) * c2 * T

def dk_f_dT(T):
    T = safe_T(T)
    return DTYPE(1.517e-2) - DTYPE(2.0) * DTYPE(1.741e-6) * T

def dmu_f_dT(T):
    T = safe_T(T)
    mu = mu_f_fun(T)
    return mu * (-DTYPE(754.1)) / (T * T + EPS)

def div_k_grad_scalar(k, dk_dT_fun, T, Tx, Ty, Txx, Tyy):
    return k * (Txx + Tyy) + dk_dT_fun(T) * (Tx * Tx + Ty * Ty)

def div_mu_grad_component(mu, dmu_dT_fun, Tf, Tfx, Tfy, ux, uy, uxx, uyy):
    return mu * (uxx + uyy) + dmu_dT_fun(Tf) * (Tfx * ux + Tfy * uy)

# ============================================================
# Residual scales
# ============================================================
D_MAX    = jnp.maximum(D_fuel, D_fluid)
SIG_REF  = jnp.maximum(jnp.abs(sigma_af_fuel(T_INLET)), jnp.abs(sigma_af_fluid(T_INLET)))
K_S_REF  = k_s_fun(T_INLET)
K_F_REF  = k_f_fun(T_INLET)
CP_S_REF = cp_s_fun(T_INLET)
CP_F_REF = cp_f_fun(T_INLET)
MU_REF   = mu_f_fun(T_INLET)

FLUX_SCALE = jnp.maximum(
    K_S_REF * TS_VAL_SCALE / (Lx_solid + EPS),
    K_F_REF * TF_VAL_SCALE / (Lx_fluid + EPS),
) + EPS

PHI_TIME_REF  = PHI_VAL_SCALE / (v_neu * (T_end + EPS))
PHI_DIFF_REF  = D_MAX * PHI_VAL_SCALE / (Lmin**2 + EPS)
PHI_REAC_REF  = SIG_REF * PHI_VAL_SCALE
PHI_RES_SCALE = PHI_TIME_REF + PHI_DIFF_REF + PHI_REAC_REF + EPS

TS_TIME_REF  = rho_s * CP_S_REF * TS_VAL_SCALE / (T_end + EPS)
TS_DIFF_REF  = K_S_REF * TS_VAL_SCALE / (Lx_solid**2 + EPS)
TS_SRC_REF   = POWER_COEF * PHI_VAL_SCALE
TS_RES_SCALE = TS_TIME_REF + TS_DIFF_REF + TS_SRC_REF + EPS

CONT_SCALE = jnp.maximum(
    U_VAL_SCALE / (Lx_fluid + EPS),
    V_VAL_SCALE / (Ly_len + EPS),
) + EPS

MOMX_TIME_REF  = rho_f * U_VAL_SCALE / (T_end + EPS)
MOMX_ADV_REF   = rho_f * (
    U_VAL_SCALE * U_VAL_SCALE / (Lx_fluid + EPS)
    + V_VAL_SCALE * U_VAL_SCALE / (Ly_len + EPS)
)
MOMX_P_REF     = P_VAL_SCALE / (Lx_fluid + EPS)
MOMX_VIS_REF   = MU_REF * (
    U_VAL_SCALE / (Lx_fluid**2 + EPS)
    + U_VAL_SCALE / (Ly_len**2 + EPS)
)
MOMX_RES_SCALE = MOMX_TIME_REF + MOMX_ADV_REF + MOMX_P_REF + MOMX_VIS_REF + EPS

MOMY_TIME_REF  = rho_f * V_VAL_SCALE / (T_end + EPS)
MOMY_ADV_REF   = rho_f * (
    U_VAL_SCALE * V_VAL_SCALE / (Lx_fluid + EPS)
    + V_VAL_SCALE * V_VAL_SCALE / (Ly_len + EPS)
)
MOMY_P_REF     = P_VAL_SCALE / (Ly_len + EPS)
MOMY_VIS_REF   = MU_REF * (
    V_VAL_SCALE / (Lx_fluid**2 + EPS)
    + V_VAL_SCALE / (Ly_len**2 + EPS)
)
MOMY_RES_SCALE = MOMY_TIME_REF + MOMY_ADV_REF + MOMY_P_REF + MOMY_VIS_REF + EPS

TF_TIME_REF  = rho_f * CP_F_REF * TF_VAL_SCALE / (T_end + EPS)
TF_ADV_REF   = rho_f * CP_F_REF * (
    U_VAL_SCALE * TF_VAL_SCALE / (Lx_fluid + EPS)
    + V_VAL_SCALE * TF_VAL_SCALE / (Ly_len + EPS)
)
TF_DIFF_REF  = K_F_REF * TF_VAL_SCALE * (
    1.0 / (Lx_fluid**2 + EPS)
    + 1.0 / (Ly_len**2 + EPS)
)
TF_RES_SCALE = TF_TIME_REF + TF_ADV_REF + TF_DIFF_REF + EPS

# ============================================================
# Targets / ICs
# ============================================================
@jax.jit
def phi_ic(y):
    return DTYPE(2.0) * jnp.cos(DTYPE(3.14) * (y - DTYPE(0.375)) / DTYPE(0.75))

def inlet_v_profile(x):
    return DTYPE(0.4) * jnp.ones_like(x)

# ============================================================
# Network
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]

    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def forward6(params, xyt):
    return mlp_apply(params, xyt[None, :])[0, :]

# ============================================================
# Sampling
# ============================================================
def sample_box(key, n, x0, x1, y0, y1, t0, t1):
    k1, k2, k3 = random.split(key, 3)
    x = random.uniform(k1, (n, 1), minval=DTYPE(x0), maxval=DTYPE(x1), dtype=DTYPE)
    y = random.uniform(k2, (n, 1), minval=DTYPE(y0), maxval=DTYPE(y1), dtype=DTYPE)
    t = random.uniform(k3, (n, 1), minval=DTYPE(t0), maxval=DTYPE(t1), dtype=DTYPE)
    return jnp.concatenate([x, y, t], axis=1)

def sample_boundary_x(key, n, x_fixed, y0, y1, t0, t1):
    k1, k2 = random.split(key, 2)
    y = random.uniform(k1, (n, 1), minval=DTYPE(y0), maxval=DTYPE(y1), dtype=DTYPE)
    t = random.uniform(k2, (n, 1), minval=DTYPE(t0), maxval=DTYPE(t1), dtype=DTYPE)
    x = DTYPE(x_fixed) * jnp.ones_like(y)
    return jnp.concatenate([x, y, t], axis=1)

def sample_boundary_y(key, n, y_fixed, x0, x1, t0, t1):
    k1, k2 = random.split(key, 2)
    x = random.uniform(k1, (n, 1), minval=DTYPE(x0), maxval=DTYPE(x1), dtype=DTYPE)
    t = random.uniform(k2, (n, 1), minval=DTYPE(t0), maxval=DTYPE(t1), dtype=DTYPE)
    y = DTYPE(y_fixed) * jnp.ones_like(x)
    return jnp.concatenate([x, y, t], axis=1)

def sample_interface(key, n):
    return sample_boundary_x(key, n, Ls, y_min, y_max, t_min, t_max)

def sample_ic(key, n):
    k1, k2 = random.split(key, 2)
    x = random.uniform(k1, (n, 1), minval=x_min, maxval=x_max, dtype=DTYPE)
    y = random.uniform(k2, (n, 1), minval=y_min, maxval=y_max, dtype=DTYPE)
    t = t_min * jnp.ones_like(x)
    return jnp.concatenate([x, y, t], axis=1)

# ============================================================
# Derivatives / residuals
# ============================================================
def eval_fields_and_derivs(params, X):
    out = vmap(lambda z: forward6(params, z))(X)
    J   = vmap(jacrev(lambda z: forward6(params, z)))(X)
    H   = vmap(hessian(lambda z: forward6(params, z)))(X)
    return out, J, H

def residuals_all(params, X):
    out, J, H = eval_fields_and_derivs(params, X)

    phi = out[:, 0]
    Ts  = out[:, 1]
    u   = out[:, 2]
    v   = out[:, 3]
    p   = out[:, 4]
    Tf  = out[:, 5]

    x = X[:, 0]
    is_solid = (x <= Ls)
    m_s = is_solid.astype(DTYPE)
    m_f = (x > Ls).astype(DTYPE)

    phi_t  = J[:, 0, 2]
    phi_xx = H[:, 0, 0, 0]
    phi_yy = H[:, 0, 1, 1]
    lap_phi = phi_xx + phi_yy

    Ts_x  = J[:, 1, 0]
    Ts_y  = J[:, 1, 1]
    Ts_t  = J[:, 1, 2]
    Ts_xx = H[:, 1, 0, 0]
    Ts_yy = H[:, 1, 1, 1]

    u_x, u_y, u_t = J[:, 2, 0], J[:, 2, 1], J[:, 2, 2]
    v_x, v_y, v_t = J[:, 3, 0], J[:, 3, 1], J[:, 3, 2]
    p_x, p_y      = J[:, 4, 0], J[:, 4, 1]
    Tf_x, Tf_y, Tf_t = J[:, 5, 0], J[:, 5, 1], J[:, 5, 2]

    u_xx, u_yy   = H[:, 2, 0, 0], H[:, 2, 1, 1]
    v_xx, v_yy   = H[:, 3, 0, 0], H[:, 3, 1, 1]
    Tf_xx, Tf_yy = H[:, 5, 0, 0], H[:, 5, 1, 1]

    T_for_sigma = jnp.where(is_solid, Ts, Tf)
    D_here      = jnp.where(is_solid, D_fuel, D_fluid)
    sig_here    = jnp.where(is_solid, sigma_af_fuel(T_for_sigma), sigma_af_fluid(T_for_sigma))

    r_phi_raw = (DTYPE(1.0) / v_neu) * phi_t - D_here * lap_phi - sig_here * phi
    r_phi = r_phi_raw / PHI_RES_SCALE

    div_k_grad_Ts = div_k_grad_scalar(k_s_fun(Ts), dk_s_dT, Ts, Ts_x, Ts_y, Ts_xx, Ts_yy)
    r_Ts_raw = rho_s * cp_s_fun(Ts) * Ts_t - div_k_grad_Ts - POWER_COEF * phi
    r_Ts = r_Ts_raw / TS_RES_SCALE

    r_cont_raw = u_x + v_y
    r_cont = r_cont_raw / CONT_SCALE

    mu_here = mu_f_fun(Tf)
    visc_u = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, u_x, u_y, u_xx, u_yy)
    visc_v = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, v_x, v_y, v_xx, v_yy)

    r_u_raw = rho_f * (u_t + u * u_x + v * u_y) + p_x - visc_u
    r_v_raw = rho_f * (v_t + u * v_x + v * v_y) + p_y - visc_v

    cp_here = cp_f_fun(Tf)
    div_k_grad_Tf = div_k_grad_scalar(k_f_fun(Tf), dk_f_dT, Tf, Tf_x, Tf_y, Tf_xx, Tf_yy)
    r_Tf_raw = rho_f * cp_here * (Tf_t + u * Tf_x + v * Tf_y) - div_k_grad_Tf

    r_u  = r_u_raw / MOMX_RES_SCALE
    r_v  = r_v_raw / MOMY_RES_SCALE
    r_Tf = r_Tf_raw / TF_RES_SCALE

    return r_phi, r_Ts, r_cont, r_u, r_v, r_Tf, m_s, m_f

# ============================================================
# Loss pieces
# ============================================================
def mse(x):
    return jnp.mean(x * x)

def pde_loss(params, X_phi, Xs, Xf, w_pde_phi, w_pde_ts, w_pde_cont, w_pde_ru, w_pde_rv, w_pde_rtf):
    rphi_all, _, _, _, _, _, _, _ = residuals_all(params, X_phi)
    _, rTs_s, _, _, _, _, _, _ = residuals_all(params, Xs)
    _, _, rcont_f, ru_f, rv_f, rTf_f, _, _ = residuals_all(params, Xf)

    lpde_phi = mse(rphi_all)
    lpde_s   = mse(rTs_s)
    lcont    = mse(rcont_f)
    lru      = mse(ru_f)
    lrv      = mse(rv_f)
    lrtf     = mse(rTf_f)

    total = (
        DTYPE(w_pde_phi) * lpde_phi
        + DTYPE(w_pde_ts) * lpde_s
        + DTYPE(w_pde_cont) * lcont
        + DTYPE(w_pde_ru) * lru
        + DTYPE(w_pde_rv) * lrv
        + DTYPE(w_pde_rtf) * lrtf
    )
    return total, (lpde_phi, lpde_s, lcont, lru, lrv, lrtf)

def bc_phi_loss(params, X_left, X_right, X_bot, X_top):
    outL = mlp_apply(params, X_left)
    outR = mlp_apply(params, X_right)
    outB = mlp_apply(params, X_bot)
    outT = mlp_apply(params, X_top)

    phiL = outL[:, 0]
    phiR = outR[:, 0]
    phiB = outB[:, 0]
    phiT = outT[:, 0]

    yL, tL = X_left[:, 1], X_left[:, 2]
    yR, tR = X_right[:, 1], X_right[:, 2]
    tB = X_bot[:, 2]
    tT = X_top[:, 2]

    return (
        mse((phiL - phi_left_target(yL, tL)) / PHI_VAL_SCALE)
        + mse((phiR - PHI_RIGHT_VAL) / PHI_VAL_SCALE)
        + mse((phiB - phi_y_target(tB)) / PHI_VAL_SCALE)
        + mse((phiT - phi_y_target(tT)) / PHI_VAL_SCALE)
    )

def bc_solid_loss(params, X_x0):
    def Ts_fun(z):
        return forward6(params, z)[1]
    Tsx_x0 = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_x0)
    return mse(Tsx_x0 / GRAD_TSX_SCALE)

def bc_fluid_loss(params, X_inlet, X_outlet):
    out_in = mlp_apply(params, X_inlet)
    out_out = mlp_apply(params, X_outlet)

    u_in, v_in, Tf_in = out_in[:, 2], out_in[:, 3], out_in[:, 5]
    p_out = out_out[:, 4]

    x_in = X_inlet[:, 0]
    v_tar = inlet_v_profile(x_in)

    return (
        mse((u_in - DTYPE(0.0)) / U_VAL_SCALE)
        + mse((v_in - v_tar) / V_VAL_SCALE)
        + mse((Tf_in - T_INLET) / TF_VAL_SCALE)
        + mse((p_out - P_OUTLET) / P_VAL_SCALE)
    )

def interface_loss(params, X_if):
    out_if = mlp_apply(params, X_if)
    Ts_if, Tf_if = out_if[:, 1], out_if[:, 5]

    def Ts_fun(z):
        return forward6(params, z)[1]

    def Tf_fun(z):
        return forward6(params, z)[5]

    Tsx_if = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_if)
    Tfx_if = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_if)

    q_s = -k_s_fun(Ts_if) * Tsx_if
    q_f = -k_f_fun(Tf_if) * Tfx_if

    return (
        mse((Ts_if - Tf_if) / T_IF_VAL_SCALE)
        + mse((q_s - q_f) / FLUX_SCALE)
    )

def ic_loss(params, X_ic):
    out0 = mlp_apply(params, X_ic)
    phi0 = out0[:, 0]
    Ts0  = out0[:, 1]
    u0   = out0[:, 2]
    v0   = out0[:, 3]
    Tf0  = out0[:, 5]

    x0 = X_ic[:, 0]
    y0 = X_ic[:, 1]
    m_s0 = (x0 <= Ls).astype(DTYPE)
    m_f0 = (x0 > Ls).astype(DTYPE)

    return (
        mse((phi0 - phi_ic(y0)) / PHI_VAL_SCALE)
        + mse(m_s0 * ((Ts0 - T_INLET) / TS_VAL_SCALE))
        + mse(m_f0 * ((u0  - FLUID_U_IC) / U_VAL_SCALE))
        + mse(m_f0 * ((v0  - FLUID_V_IC) / V_VAL_SCALE))
        + mse(m_f0 * ((Tf0 - FLUID_T_IC) / TF_VAL_SCALE))
    )

# ============================================================
# Total loss
# ============================================================
@partial(jax.jit, static_argnames=())
def loss_fn(
    params,
    X_pde_phi, X_pde_s, X_pde_f,
    X_phi_left, X_phi_right, X_phi_y0, X_phi_y1,
    X_Ts_x0,
    X_f_inlet, X_f_outlet,
    X_if, X_ic,
    w_pde_phi, w_pde_ts, w_pde_cont, w_pde_ru, w_pde_rv, w_pde_rtf,
    w_bc_phi, w_bc_solid, w_bc_fluid, w_if, w_ic
):
    lpde_total, (lpde_phi, lpde_s, lcont, lru, lrv, lrtf) = pde_loss(
        params, X_pde_phi, X_pde_s, X_pde_f,
        w_pde_phi, w_pde_ts, w_pde_cont, w_pde_ru, w_pde_rv, w_pde_rtf
    )

    lbc_phi = bc_phi_loss(params, X_phi_left, X_phi_right, X_phi_y0, X_phi_y1)
    lbc_sol = bc_solid_loss(params, X_Ts_x0)
    lbc_fld = bc_fluid_loss(params, X_f_inlet, X_f_outlet)
    lif = interface_loss(params, X_if)
    lic = ic_loss(params, X_ic)

    total = (
        lpde_total
        + DTYPE(w_bc_phi) * lbc_phi
        + DTYPE(w_bc_solid) * lbc_sol
        + DTYPE(w_bc_fluid) * lbc_fld
        + DTYPE(w_if) * lif
        + DTYPE(w_ic) * lic
    )

    return total, (lpde_phi, lpde_s, lcont, lru, lrv, lrtf, lbc_phi, lbc_sol, lbc_fld, lif, lic)

loss_and_grad = jax.jit(jax.value_and_grad(loss_fn, argnums=0, has_aux=True))

# ============================================================
# Adam
# ============================================================
def adam_init(params):
    m = tree_util.tree_map(lambda p: jnp.zeros_like(p), params)
    v = tree_util.tree_map(lambda p: jnp.zeros_like(p), params)
    return m, v

def adam_step(params, grads, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m = tree_util.tree_map(lambda m_i, g_i: beta1 * m_i + (1.0 - beta1) * g_i, m, grads)
    v = tree_util.tree_map(lambda v_i, g_i: beta2 * v_i + (1.0 - beta2) * (g_i * g_i), v, grads)

    mhat = tree_util.tree_map(lambda m_i: m_i / (1.0 - beta1 ** t), m)
    vhat = tree_util.tree_map(lambda v_i: v_i / (1.0 - beta2 ** t), v)

    params = tree_util.tree_map(
        lambda p_i, mh_i, vh_i: p_i - lr * mh_i / (jnp.sqrt(vh_i) + eps),
        params, mhat, vhat
    )
    return params, m, v

def flatten_params(params):
    flat = []
    for lyr in params:
        flat.append(np.asarray(lyr["W"]).reshape(-1))
        flat.append(np.asarray(lyr["b"]).reshape(-1))
    return np.concatenate(flat)

def _rms(x):
    return float(jnp.sqrt(jnp.mean(x * x) + 1e-30))

# ============================================================
# Train Adam
# ============================================================
def train_adam(
    seed=0,
    hidden_dim=35,
    num_hidden=3,
    max_iters=5000,
    print_every=100,
    lr=1e-4,

    n_pde_phi=4096,
    n_pde_s=4096,
    n_pde_f=4096,

    n_phi_left=512,
    n_phi_right=256,
    n_phi_y0=256,
    n_phi_y1=256,

    n_Ts_x0=512,

    n_f_inlet=512,
    n_f_outlet=512,

    n_if=1024,
    n_ic=1024,

    # match current SQP objective weights
    w_pde_phi=5.0,
    w_pde_ts=5.0,
    w_pde_cont=20.0,
    w_pde_ru=40.0,
    w_pde_rv=40.0,
    w_pde_rtf=5.0,

    w_bc_phi=10.0,
    w_bc_solid=10.0,
    w_bc_fluid=10.0,
    w_if=10.0,
    w_ic=10.0,
):
    key = random.PRNGKey(seed)

    layer_sizes = [3] + [hidden_dim] * num_hidden + [6]
    key, k0 = random.split(key)
    params = init_mlp_params(k0, layer_sizes)

    m, v = adam_init(params)

    print("Adam training matching CURRENT SQP benchmark")
    print("layer_sizes =", layer_sizes)

    t0_clock = time.time()

    for it in range(1, max_iters + 1):
        key, *sub = random.split(key, 13)
        (
            kphi, ks, kf,
            k1, k2, k3, k4,
            k5,
            k8, k9,
            kif, kic
        ) = sub

        X_pde_phi = sample_box(kphi, n_pde_phi, x_min, x_max, y_min, y_max, t_min, t_max)
        X_pde_s   = sample_box(ks,   n_pde_s,   x_min, Ls,   y_min, y_max, t_min, t_max)
        X_pde_f   = sample_box(kf,   n_pde_f,   Ls,   x_max, y_min, y_max, t_min, t_max)

        X_phi_left  = sample_boundary_x(k1, n_phi_left,  x_min, y_min, y_max, t_min, t_max)
        X_phi_right = sample_boundary_x(k2, n_phi_right, x_max, y_min, y_max, t_min, t_max)
        X_phi_y0    = sample_boundary_y(k3, n_phi_y0,    y_min, x_min, x_max, t_min, t_max)
        X_phi_y1    = sample_boundary_y(k4, n_phi_y1,    y_max, x_min, x_max, t_min, t_max)

        X_Ts_x0 = sample_boundary_x(k5, n_Ts_x0, x_min, y_min, y_max, t_min, t_max)

        X_f_inlet  = sample_boundary_y(k8, n_f_inlet,  y_min, Ls, x_max, t_min, t_max)
        X_f_outlet = sample_boundary_y(k9, n_f_outlet, y_max, Ls, x_max, t_min, t_max)

        X_if = sample_interface(kif, n_if)
        X_ic = sample_ic(kic, n_ic)

        (loss_val, aux), grads = loss_and_grad(
            params,
            X_pde_phi, X_pde_s, X_pde_f,
            X_phi_left, X_phi_right, X_phi_y0, X_phi_y1,
            X_Ts_x0,
            X_f_inlet, X_f_outlet,
            X_if, X_ic,
            DTYPE(w_pde_phi), DTYPE(w_pde_ts), DTYPE(w_pde_cont),
            DTYPE(w_pde_ru), DTYPE(w_pde_rv), DTYPE(w_pde_rtf),
            DTYPE(w_bc_phi), DTYPE(w_bc_solid), DTYPE(w_bc_fluid),
            DTYPE(w_if), DTYPE(w_ic),
        )

        params, m, v = adam_step(params, grads, m, v, it, lr)

        if it % print_every == 0:
            lpde_phi, lpde_s, lcont, lru, lrv, lrtf, lbc_phi, lbc_sol, lbc_fld, lif, lic = aux

            rphi_all, _, _, _, _, _, _, _ = residuals_all(params, X_pde_phi)
            _, rTs_s, _, _, _, _, _, _ = residuals_all(params, X_pde_s)
            _, _, rcont_f, ru_f, rv_f, rTf_f, _, _ = residuals_all(params, X_pde_f)

            print(
                f"[Adam it={it}] loss={float(loss_val):.3e} "
                f"lpde_phi={float(lpde_phi):.3e} lpde_s={float(lpde_s):.3e} "
                f"cont={float(lcont):.3e} ru={float(lru):.3e} rv={float(lrv):.3e} rTf={float(lrtf):.3e} "
                f"lbc_phi={float(lbc_phi):.3e} lbc_sol={float(lbc_sol):.3e} "
                f"lbc_fld={float(lbc_fld):.3e} lif={float(lif):.3e} lic={float(lic):.3e}"
            )
            print(
                "Scaled PDE RMS:",
                "phi", _rms(rphi_all),
                "Ts_s", _rms(rTs_s),
                "cont_f", _rms(rcont_f),
                "ru_f", _rms(ru_f),
                "rv_f", _rms(rv_f),
                "rTf_f", _rms(rTf_f),
            )

    print(f"[done] elapsed={time.time() - t0_clock:.2f}s")
    return params

# ============================================================
# Main
# ============================================================
def main():
    params = train_adam(
        seed=0,
        hidden_dim=64,
        num_hidden=3,
        max_iters=10000,
        print_every=100,
        lr=1e-4,
    )
    theta = flatten_params(params)
    np.save("adam_pinn_match_current_sqp.npy", theta)
    print("Saved flat theta -> adam_pinn_match_current_sqp.npy")

if __name__ == "__main__":
    main()

Adam training matching CURRENT SQP benchmark
layer_sizes = [3, 64, 64, 64, 6]


2026-03-30 07:10:53.988521: W external/xla/xla/stream_executor/cuda/cuda_command_buffer.cc:725] Retry CUDA graph instantiation after OOM error
E0330 07:10:54.000149 1741931 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Underlying backend ran out of memory trying to instantiate command buffer with 12 (total of 21 alive graphs in the process). You can try to (a) Give more memory to the driver by reducing XLA_CLIENT_MEM_FRACTION (b) Disable command buffers with 'XLA_FLAGS=--xla_gpu_enable_command_buffer=' (empty set). Original error: Failed to instantiate CUDA graph: CUDA_ERROR_OUT_OF_MEMORY: out of memory


XlaRuntimeError: RESOURCE_EXHAUSTED: Underlying backend ran out of memory trying to instantiate command buffer with 12 (total of 21 alive graphs in the process). You can try to (a) Give more memory to the driver by reducing XLA_CLIENT_MEM_FRACTION (b) Disable command buffers with 'XLA_FLAGS=--xla_gpu_enable_command_buffer=' (empty set). Original error: Failed to instantiate CUDA graph: CUDA_ERROR_OUT_OF_MEMORY: out of memory

In [ ]:
print("phi_true.shape =", phi_true.shape)

# assume phi_true is 3D
a, b, c = phi_true.shape
print("axis sizes:", a, b, c)

def stats(name, arr):
    print(name)
    print("  shape   :", arr.shape)
    print("  mean abs:", np.mean(np.abs(arr)))
    print("  min/max :", np.min(arr), np.max(arr))

# all six outer slices
stats("axis1 first  phi_true[0,:,:]",   phi_true[0, :, :])
stats("axis1 last   phi_true[-1,:,:]",  phi_true[-1, :, :])

stats("axis2 first  phi_true[:,0,:]",   phi_true[:, 0, :])
stats("axis2 last   phi_true[:,-1,:]",  phi_true[:, -1, :])

stats("axis3 first  phi_true[:,:,0]",   phi_true[:, :, 0])
stats("axis3 last   phi_true[:,:,-1]",  phi_true[:, :, -1])

phi_true.shape = (17, 20, 64)
axis sizes: 17 20 64
axis1 first  phi_true[0,:,:]
  shape   : (20, 64)
  mean abs: 1.2736294023845356
  min/max : 0.05063449870848282 1.998796676955403
axis1 last   phi_true[-1,:,:]
  shape   : (20, 64)
  mean abs: 2.7600419738549364
  min/max : 0.04744169422884767 10.483047280042607
axis2 first  phi_true[:,0,:]
  shape   : (17, 64)
  mean abs: 3.487378040282388
  min/max : 0.05063449870848282 10.483047280042607
axis2 last   phi_true[:,-1,:]
  shape   : (17, 64)
  mean abs: 0.1759052020626532
  min/max : 0.019286873502372324 1.998796676955403
axis3 first  phi_true[:,:,0]
  shape   : (17, 20)
  mean abs: 0.8020025868223284
  min/max : 0.05063449870848282 1.6646823997930156
axis3 last   phi_true[:,:,-1]
  shape   : (17, 20)
  mean abs: 0.5493111143768257
  min/max : 0.05063449870848349 0.8793695867465514
